In [20]:
import scipy.io as sio
import os

# Carpeta donde están los archivos .mat
folder = "stroke-rehab"

# Lista de archivos .mat dentro de la carpeta
mat_files = [f for f in os.listdir(folder) if f.endswith(".mat")]

# Diccionario donde guardaremos los datos cargados
data = {}

for filename in mat_files:
    filepath = os.path.join(folder, filename)
    data[filename] = sio.loadmat(filepath)

print("Carga completada.")


Carga completada.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def prepare_eeg_dataset_windowed(signals, triggers, fs=256):
    """
    Prepara datos EEG en ventanas de 1.5s.

    Cada trial original dura 6 segundos útiles
    (desde el segundo 2 al 8 del trigger),
    y se divide en 4 ventanas de 1.5s.

    Parámetros:
    -----------
    signals : np.ndarray
        Shape = (n_samples, n_channels)
    triggers : np.ndarray
        Shape = (n_samples, 1) o (n_samples,)
    fs : int
        Frecuencia de muestreo

    Retorna:
    --------
    X : np.ndarray
        Shape = (n_windows, window_samples, n_channels)
    y : np.ndarray
        Shape = (n_windows,)
    """

    triggers = triggers.flatten()

    # Configuración temporal
    trigger_duration = 8 * fs
    offset_start = 2 * fs
    useful_duration = 6 * fs

    window_duration = 1.5 * fs   # 1.5 segundos
    window_duration = int(window_duration)

    # Número de ventanas en 6 segundos
    n_windows_per_trial = useful_duration // window_duration

    X, y = [], []

    i = 0
    n = len(triggers)

    while i < n:
        if triggers[i] in [-1, 1]:
            current_trigger = triggers[i]

            start_trigger = i
            while i < n and triggers[i] == current_trigger:
                i += 1
            end_trigger = i

            if (end_trigger - start_trigger) >= trigger_duration:
                start_sample = start_trigger + offset_start
                end_sample = start_sample + useful_duration

                if end_sample <= len(signals):
                    trial = signals[start_sample:end_sample]

                    label = 0 if current_trigger == -1 else 1

                    # Dividir en ventanas
                    for w in range(n_windows_per_trial):
                        ws = w * window_duration
                        we = ws + window_duration
                        window = trial[ws:we]

                        X.append(window)
                        y.append(label)

        else:
            i += 1

    X = np.array(X)
    y = np.array(y)

    return X, y


In [23]:
for k in data.keys():
    signals = data[k]['y']
    triggers = data[k]['trig']

    X, y = prepare_eeg_dataset_windowed(signals, triggers)
    np.savez(f'datos_procesados/{k}.npz', X=X, y=y)

    print(k)
    print('-'*30)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Distribución etiquetas:", np.bincount(y))
    print()


P1_post_test.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P1_post_training.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P1_pre_test.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P1_pre_training.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P2_post_test.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P2_post_training.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P2_pre_test.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P2_pre_training.mat
------------------------------
X shape: (320, 384, 16)
y shape: (320,)
Distribución etiquetas: [160 160]

P3_p